<a href="https://colab.research.google.com/github/fatimaali123-ai/flyrank-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fatimaali123-ai/flyrank-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


I will use a Random Forest classifier because the task has a yes/no observed label, `is_declining_label`. A tree-based model can capture non-linear relationships between content performance signals without requiring strong assumptions about the relationship between features and the label.

I chose Random Forest after the simpler baseline because the Week-4 baseline is a transparent hand-written rule based mainly on impressions and CTR. The goal is not to add complexity for its own sake, but to test whether a learned model can identify declining content more effectively.

I will evaluate the model as a ranking system using Precision@20 and Precision@50, because the practical question is which content should be reviewed first. I will also report the positive-class base rate for context.

The model will not use `trend_pct`, `trend_direction`, or other label-derived fields because these would leak information about the target.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I will use a grouped train/test split by `client_id`. This prevents content from the same client from appearing in both training and test sets, which gives a more honest estimate of how the model transfers across clients.

I will use an 80/20 split with a fixed random seed for reproducibility. The same test set will be used for both the Random Forest model and the Week-4 baseline score.

The split is grouped because `client_id` identifies the client context, while `content_id` is only an identifier and must not be used as a predictive feature.


In [ ]:
!git clone https://github.com/fatimaali123-ai/flyrank-internship.git

fatal: destination path 'flyrank-internship' already exists and is not an empty directory.


In [ ]:
import os

print(os.listdir("/content/flyrank-internship"))

['DATA_USE.md', 'README.md', 'notebooks', 'docs', 'work', 'Copy_of_02_your_first_readable_model.ipynb', 'SETUP.md', 'data', 'scripts', 'skills', 'outputs', '.git', 'AGENTS.md', 'LICENSE', '.github', 'submission', 'w01_research_question.ipynb', 'CLAUDE.md', 'GUIDE.md', '.gitignore', 'requirements.txt']


In [ ]:
import os

data_path = "/content/flyrank-internship/data/raw/content_refresh_anonymized.csv"

print("File exists:", os.path.exists(data_path))

File exists: True


In [ ]:
print("ALL COLUMNS:")
for i, col in enumerate(df.columns, 1):
    print(i, "->", col)

ALL COLUMNS:
1 -> content_id
2 -> client_id
3 -> search_volume
4 -> competition
5 -> competition_level
6 -> cpc
7 -> content_type
8 -> main_intent
9 -> word_count
10 -> char_count
11 -> provider_used
12 -> model_used
13 -> impressions_90d
14 -> clicks_90d
15 -> pageviews_90d
16 -> sessions_90d
17 -> users_90d
18 -> engaged_sessions_90d
19 -> ai_sessions_90d
20 -> scroll_events_90d
21 -> days_with_impressions
22 -> days_with_sessions
23 -> impressions_last_30d
24 -> clicks_last_30d
25 -> sessions_last_30d
26 -> impressions_prev_30d
27 -> clicks_prev_30d
28 -> sessions_prev_30d
29 -> content_age_days
30 -> age_tier
31 -> age_tier_order
32 -> days_since_last_update
33 -> freshness_tier
34 -> word_count_tier
35 -> char_count_tier
36 -> ctr
37 -> avg_position
38 -> engagement_rate
39 -> scroll_rate
40 -> ai_traffic_pct
41 -> impression_tier
42 -> position_tier
43 -> trend_direction
44 -> trend_pct
45 -> is_declining_label


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit

RANDOM_STATE = 42

# Load data
df = pd.read_csv(
    "/content/flyrank-internship/data/raw/content_refresh_anonymized.csv"
)

print("Dataset shape:", df.shape)

# ---------------------------------------------------------
# Create the observed bootstrap decline label
# ---------------------------------------------------------

df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower().eq("down").astype(int)
)

print("\nTarget distribution:")
print(df["is_declining_label"].value_counts())

print("\nTarget rate:")
print(df["is_declining_label"].mean())

# ---------------------------------------------------------
# Grouped 80/20 train-test split
# ---------------------------------------------------------

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_idx, test_idx = next(
    gss.split(
        df,
        y=df["is_declining_label"],
        groups=df["client_id"]
    )
)

train = df.iloc[train_idx].copy()
test = df.iloc[test_idx].copy()

print("\nTrain shape:", train.shape)
print("Test shape:", test.shape)

print("\nTrain clients:", train["client_id"].nunique())
print("Test clients:", test["client_id"].nunique())

print("\nClient overlap:")
print(set(train["client_id"]).intersection(set(test["client_id"])))

print("\nTrain positive rate:", train["is_declining_label"].mean())
print("Test positive rate:", test["is_declining_label"].mean())

Dataset shape: (30000, 44)

Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Target rate:
0.5420666666666667

Train shape: (23837, 45)
Test shape: (6163, 45)

Train clients: 25
Test clients: 7

Client overlap:
set()

Train positive rate: 0.5501111717078492
Test positive rate: 0.5109524582184002


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
# =========================================================
# SECTION 3 — TRAIN RANDOM FOREST + COMPARE WITH BASELINE
# =========================================================

from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# 1. Define features
# ---------------------------------------------------------

# These columns must NOT be used as model features
excluded = [
    "content_id",
    "client_id",
    "is_declining_label",
    "trend_direction",
    "trend_pct"
]

# Only use numeric columns
feature_cols = [
    col for col in train.columns
    if col not in excluded
    and pd.api.types.is_numeric_dtype(train[col])
]

print("Number of features:", len(feature_cols))
print("\nFeatures used by the model:")
print(feature_cols)

# ---------------------------------------------------------
# 2. Prepare X and y
# ---------------------------------------------------------

X_train = train[feature_cols].copy()
X_test = test[feature_cols].copy()

y_train = train["is_declining_label"].astype(int)
y_test = test["is_declining_label"].astype(int)

# ---------------------------------------------------------
# 3. Handle missing values
# ---------------------------------------------------------

# IMPORTANT:
# Calculate medians using TRAINING data only.
# Then apply those medians to both train and test.

train_medians = X_train.median()

X_train = X_train.fillna(train_medians)
X_test = X_test.fillna(train_medians)

print("\nMissing values after filling:")
print("Train:", X_train.isna().sum().sum())
print("Test :", X_test.isna().sum().sum())

# ---------------------------------------------------------
# 4. Train Random Forest
# ---------------------------------------------------------

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

model.fit(X_train, y_train)

print("\nRandom Forest trained successfully.")

# ---------------------------------------------------------
# 5. Get model scores
# ---------------------------------------------------------

model_scores = model.predict_proba(X_test)[:, 1]

print("\nExample model scores:")
print(model_scores[:10])

# ---------------------------------------------------------
# 6. Precision@K function
# ---------------------------------------------------------

def precision_at_k(y_true, scores, k):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    top_indices = np.argsort(-scores)[:k]

    return y_true[top_indices].mean()

# ---------------------------------------------------------
# 7. Week-4 baseline
# ---------------------------------------------------------

# Week-4 rule:
# High impressions + lower CTR

baseline_scores = (
    test["impressions_90d"] *
    (1 - test["ctr"] / 100)
).fillna(0)

# ---------------------------------------------------------
# 8. Compare baseline vs model
# ---------------------------------------------------------

results = pd.DataFrame({
    "Method": [
        "Week-4 baseline",
        "Random Forest"
    ],
    "Precision@20": [
        precision_at_k(y_test, baseline_scores, 20),
        precision_at_k(y_test, model_scores, 20)
    ],
    "Precision@50": [
        precision_at_k(y_test, baseline_scores, 50),
        precision_at_k(y_test, model_scores, 50)
    ]
})

print("\n======================================")
print("MODEL VS BASELINE")
print("======================================")

display(results)

# ---------------------------------------------------------
# 9. Base rate
# ---------------------------------------------------------

print("\nTest-set positive rate:")
print(round(y_test.mean(), 4))

# ---------------------------------------------------------
# 10. Feature importance
# ---------------------------------------------------------

importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print("\n======================================")
print("TOP 10 FEATURES")
print("======================================")

display(importance.head(10))


Number of features: 29

Features used by the model:
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier_order', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']

Missing values after filling:
Train: 0
Test : 0

Random Forest trained successfully.

Example model scores:
[0.66236379 0.47672335 0.7098451  0.70598862 0.60995464 0.65073407
 0.70777511 0.58015494 0.59164041 0.64444949]

MODEL VS BASELINE


,Method,Precision@20,Precision@50
0,Week-4 baseline,0.35,0.44
1,Random Forest,1.00,1.00



Test-set positive rate:
0.511

TOP 10 FEATURES


,feature,importance
18,impressions_prev_30d,0.290325
15,impressions_last_30d,0.121944
5,impressions_90d,0.090560
13,days_with_impressions,0.076487
21,content_age_days,0.072354
25,avg_position,0.070341
22,age_tier_order,0.046069
16,clicks_last_30d,0.030218
17,sessions_last_30d,0.026865
3,word_count,0.023613


I will train a Random Forest classifier using the available pre-outcome numeric features. I will exclude identifiers and label-derived fields to prevent leakage.

The Week-4 baseline is the existing hand-written refresh score based on high impressions and lower CTR. I will calculate the baseline and Random Forest rankings on the same held-out test set.

The main evaluation metrics are Precision@20 and Precision@50 because the practical goal is to prioritize a small number of content items for review. The model will only be considered an improvement if it performs better than the baseline on the same test set.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# =========================================================
# SECTION 4 — ERROR ANALYSIS
# =========================================================

# Create a results dataframe for every test example

error_df = test[
    ["content_id", "client_id", "trend_direction", "trend_pct"]
].copy()

error_df["actual"] = y_test.to_numpy()
error_df["model_score"] = model_scores
error_df["model_prediction"] = (model_scores >= 0.5).astype(int)

# ---------------------------------------------------------
# 1. False positives and false negatives
# ---------------------------------------------------------

false_positives = error_df[
    (error_df["actual"] == 0) &
    (error_df["model_prediction"] == 1)
].copy()

false_negatives = error_df[
    (error_df["actual"] == 1) &
    (error_df["model_prediction"] == 0)
].copy()

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

# ---------------------------------------------------------
# 2. Classification summary
# ---------------------------------------------------------

true_positives = (
    (error_df["actual"] == 1) &
    (error_df["model_prediction"] == 1)
).sum()

true_negatives = (
    (error_df["actual"] == 0) &
    (error_df["model_prediction"] == 0)
).sum()

print("\nClassification counts:")
print("True positives :", true_positives)
print("True negatives :", true_negatives)
print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

# ---------------------------------------------------------
# 3. Top-ranked predictions
# ---------------------------------------------------------

top_20 = error_df.sort_values(
    "model_score",
    ascending=False
).head(20)

print("\nTop 20 model-ranked items:")
display(top_20)

# ---------------------------------------------------------
# 4. Inspect false positives
# ---------------------------------------------------------

print("\nFalse positives:")
display(
    false_positives
    .sort_values("model_score", ascending=False)
    .head(10)
)

# ---------------------------------------------------------
# 5. Inspect false negatives
# ---------------------------------------------------------

print("\nFalse negatives:")
display(
    false_negatives
    .sort_values("model_score", ascending=True)
    .head(10)
)

# ---------------------------------------------------------
# 6. Compare model confidence
# ---------------------------------------------------------

print("\nModel score summary:")
print(error_df["model_score"].describe())

False positives: 1016
False negatives: 740

Classification counts:
True positives : 2409
True negatives : 1998
False positives: 1016
False negatives: 740

Top 20 model-ranked items:


,content_id,client_id,trend_direction,trend_pct,actual,model_score,model_prediction
5848,content_4b40bc3fc714,client_bdd2d3af3a,down,-100.0,1,0.842317,1
8184,content_99f7f3940ed4,client_8527a891e2,down,-100.0,1,0.834037,1
22896,content_9bbf99b4dc21,client_434c9b5ae5,down,-100.0,1,0.826447,1
10171,content_72f65ab80b1d,client_bdd2d3af3a,down,-100.0,1,0.823127,1
22596,content_f05beee7738a,client_434c9b5ae5,down,-100.0,1,0.815619,1
13454,content_0b50482209cd,client_bdd2d3af3a,down,-100.0,1,0.810953,1
11994,content_e3bf6539b4ba,client_bdd2d3af3a,down,-100.0,1,0.808857,1
11578,content_e6be4a754962,client_434c9b5ae5,down,-100.0,1,0.808728,1
2975,content_639b9e356f82,client_bdd2d3af3a,down,-100.0,1,0.808526,1
10679,content_a61f639a5927,client_f369cb89fc,down,-100.0,1,0.802503,1



False positives:


,content_id,client_id,trend_direction,trend_pct,actual,model_score,model_prediction
20736,content_41baf0722ad9,client_8527a891e2,stable,-14.3,0,0.743062,1
2357,content_8f1409b2674e,client_8527a891e2,stable,-17.9,0,0.742569,1
2488,content_204b729683f6,client_f369cb89fc,stable,-13.2,0,0.741534,1
28718,content_ef6e7d7cfe15,client_8527a891e2,stable,11.4,0,0.740970,1
12332,content_4d9f36001f06,client_8527a891e2,stable,-17.0,0,0.740500,1
11887,content_ce59581533ca,client_8527a891e2,stable,-8.0,0,0.731189,1
18326,content_459756cca996,client_f369cb89fc,stable,-18.8,0,0.730754,1
19620,content_f13a486a08f2,client_8527a891e2,stable,8.3,0,0.729838,1
11061,content_0b47dae0c7f9,client_8527a891e2,stable,-13.3,0,0.729190,1
1517,content_816d77e36e14,client_8527a891e2,stable,-19.1,0,0.727984,1



False negatives:


,content_id,client_id,trend_direction,trend_pct,actual,model_score,model_prediction
18929,content_f3ca73f0f3f3,client_e629fa6598,down,-33.3,1,0.231206,0
27575,content_284df888e0cb,client_e629fa6598,down,-25.0,1,0.238148,0
3709,content_6e6e8c6fcd2a,client_e629fa6598,down,-20.8,1,0.245879,0
14282,content_adb76cf9c286,client_e629fa6598,down,-67.2,1,0.255034,0
2208,content_827b209fa167,client_4e07408562,down,-23.0,1,0.263766,0
24864,content_8ab9f1033843,client_e629fa6598,down,-27.6,1,0.263887,0
17727,content_82074408a03e,client_e629fa6598,down,-32.8,1,0.263951,0
5017,content_d8a3c9484a1b,client_e629fa6598,down,-47.7,1,0.269259,0
252,content_aba4b4460e47,client_e629fa6598,down,-49.7,1,0.273005,0
11655,content_cea79ef51519,client_f369cb89fc,down,-35.6,1,0.276089,0



Model score summary:
count    6163.000000
mean        0.503009
std         0.190732
min         0.004171
25%         0.396045
50%         0.539100
75%         0.659421
max         0.842317
Name: model_score, dtype: float64


The Random Forest substantially outperformed the Week-4 baseline on the held-out test set. The baseline achieved Precision@20 of 0.35 and Precision@50 of 0.44, while the Random Forest achieved 1.00 on both metrics.

The test-set positive rate was 0.511, so the perfect top-k precision is much higher than the overall prevalence of declining content.

The most important features were `impressions_prev_30d`, `impressions_last_30d`, `impressions_90d`, `days_with_impressions`, and `content_age_days`. This suggests that recent and historical traffic patterns and content age are strong signals for distinguishing declining content.

Because the model achieved perfect top-k precision, I will inspect the top-ranked predictions and the model's false positives and false negatives rather than assuming that the result automatically means the model generalizes perfectly. I will also check whether the strongest signals could be acting as proxies for the way the bootstrap decline label was constructed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.